In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
groq_client = OpenAI(
    base_url = "https://api.groq.com/openai/v1",
    api_key = os.getenv("GROQ_API_KEY")
)
or_client = OpenAI(
    base_url= "https://openrouter.ai/api/v1",
    api_key = os.getenv("OPENROUTER_API_KEY")
)

In [3]:
from google import genai
from google.genai import types
genai_client = genai.Client(api_key=os.getenv("GENAI_API_KEY"))

In [4]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [8]:
from sqlitesearch import TextSearchIndex
index = TextSearchIndex(
    text_fields= ['question','section','answer'],
    keyword_fields=['course'],
    db_path="database/FAQ.db"
)

In [14]:
def search(query: str) -> dict[str,str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    boost_dict = {'question':3.0, 'section':.5}
    filter_dict = {'course': 'llm-zoomcamp'}
    
    return index.search(
        query,
        num_results = 5,
        boost_dict=boost_dict,
        filter_dict= filter_dict
    )

In [15]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [17]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [42]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

In [48]:
chat_interface = IPythonChatInterface()
callback= DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools = agent_tools,
    developer_prompt = instructions,
    chat_interface= chat_interface,
    llm_client=OpenAIClient(client=or_client,model="cohere/north-mini-code:free")
)

In [ ]:
result = runner.loop(
    prompt="How do I run Olama locally",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


-> Response received


/workspaces/DataTalks_LLms/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'free'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [51]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally? my mom is sick and she needs it for her class', role='user', phase=None, type=None),
 ResponseReasoningItem(id='rs_tmp_zst2y8dg1ko', summary=[], type='reasoning', content=[Content(text='The user is asking about running Olama locally and mentions their mother needs it for her class. This seems to be a technical question about software installation and usage. I should search for information about Olama to un